In [ ]:
# ───────────────────────────────────────────────────────────────────
# 1) User settings: adjust these to match your data
# ───────────────────────────────────────────────────────────────────
FILENAME         = '/content/A5 S1C-09 Casing-Insulation Debond G-22-23 Rx.csv'    # your CSV file
TIME_UNIT        = 'ms'          # original time unit in the CSV: 'µs', 'ms', or 's'
AMPLITUDE_UNIT   = 'V'           # e.g. 'V', 'a.u.', etc.
US_VELOCITY      = 1500.0        # ultrasonic speed in solid (m/s)
DISTANCE_UNIT    = 'mm'          # desired output distance unit: 'm' or 'mm'

# automatically compute scale factors from unit strings
if TIME_UNIT == 'µs':
    TIME_SCALE = 1e-6
elif TIME_UNIT == 'ms':
    TIME_SCALE = 1e-3
else:
    TIME_SCALE = 1.0

DIST_SCALE = 1e3 if DISTANCE_UNIT == 'mm' else 1.0
FREQ_UNIT  = 'kHz'
# ───────────────────────────────────────────────────────────────────
# 2) Imports
# ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.fft import fft, fftfreq

# ───────────────────────────────────────────────────────────────────
# 3) Load and convert data
# ───────────────────────────────────────────────────────────────────
df        = pd.read_csv("/content/Bonded zone A5 S1C-09 CPCDC Rx.csv")
time_raw  = df['time'].values
amp       = df['amplitude'].values

# convert raw time to seconds
time_s    = time_raw * TIME_SCALE

# ───────────────────────────────────────────────────────────────────
# 4) Basic statistics
# ───────────────────────────────────────────────────────────────────
T0        = 0.0
mean_amp  = np.mean(amp)
std_amp   = np.std(amp)

# ───────────────────────────────────────────────────────────────────
# 5) Helper to find peak in window
# ───────────────────────────────────────────────────────────────────
def find_peak(t, a, tmin, tmax):
    mask = (t >= tmin) & (t <= tmax)
    if not mask.any():
        return np.nan, np.nan
    idx   = np.argmax(a[mask])
    twin  = t[mask]
    awin  = a[mask]
    return float(twin[idx]), float(awin[idx])

# windows in seconds
w1_min, w1_max = 140 * TIME_SCALE, 260 * TIME_SCALE
w2_min, w2_max = 300 * TIME_SCALE, 400 * TIME_SCALE
w3_min, w3_max = 400 * TIME_SCALE, 600 * TIME_SCALE

T1, A1 = find_peak(time_s, amp, w1_min, w1_max)
Tp, Ap = find_peak(time_s, amp, w2_min, w2_max)
Tr, Ar = find_peak(time_s, amp, w3_min, w3_max)

# ───────────────────────────────────────────────────────────────────
# 6) Count cycles (peaks above mean)
# ───────────────────────────────────────────────────────────────────
peaks, _ = find_peaks(amp, height=mean_amp)
cycles   = len(peaks)

# ───────────────────────────────────────────────────────────────────
# 7) FFT for dominant frequency & magnitude
# ───────────────────────────────────────────────────────────────────
N    = len(amp)
dt   = time_s[1] - time_s[0]
yf   = fft(amp - mean_amp)
xf   = fftfreq(N, dt)

# take positive freqs and convert to kHz
mask = xf > 0
xf_khz = xf[mask] / 1e3
yf_mag = np.abs(yf[mask])

idx_dom = np.argmax(yf_mag)
fft_f   = float(xf_khz[idx_dom])
fft_M   = float(yf_mag[idx_dom])

# ───────────────────────────────────────────────────────────────────
# 8) Distance to defect
# ───────────────────────────────────────────────────────────────────
# using two-way travel: d = v * t_TOF / 2
dist_m   = US_VELOCITY * T1 / 2
dist_out = dist_m * DIST_SCALE

# ───────────────────────────────────────────────────────────────────
# 9) Time-domain plot with annotations
# ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(time_raw, amp, lw=1)

# mark & label T0, T1, Tp, Tr in original time units
for t_s, a, lbl in [
    (T0,       0,   'T0'),
    (T1,       A1,  f'T1={T1/TIME_SCALE:.1f}{TIME_UNIT}'),
    (Tp,       Ap,  f'Tp={Tp/TIME_SCALE:.1f}{TIME_UNIT}'),
    (Tr,       Ar,  f'Tr={Tr/TIME_SCALE:.1f}{TIME_UNIT}')
]:
    t_raw = t_s / TIME_SCALE
    ax.plot([t_raw], [a], 'o')
    ax.annotate(lbl, (t_raw, a), xytext=(5,5), textcoords='offset points')

# dashed lines at window edges
for edge in (140, 260, 300, 400, 600):
    ax.axvline(edge, ls='--', alpha=0.5)

ax.set_xlabel(f'Time ({TIME_UNIT})')
ax.set_ylabel(f'Amplitude ({AMPLITUDE_UNIT})')
ax.set_title('Amplitude vs. Time with Annotated Peaks')
ax.grid(True)
plt.tight_layout()
plt.show()



# ───────────────────────────────────────────────────────────────────
# 11) Print all 13 parameters with units
# ───────────────────────────────────────────────────────────────────
print("Computed Parameters:")
print(f"  1)  T₀                      = {T0:.3f} {TIME_UNIT}")
print(f"  2)  T₁ (TOF)                = {T1/TIME_SCALE:.3f} {TIME_UNIT}")
print(f"      A₁                      = {A1:.3f} {AMPLITUDE_UNIT}")
print(f"  3)  Tₚ                      = {Tp/TIME_SCALE:.3f} {TIME_UNIT}")
print(f"      Aₚ                      = {Ap:.3f} {AMPLITUDE_UNIT}")
print(f"  4)  Tᵣ                      = {Tr/TIME_SCALE:.3f} {TIME_UNIT}")
print(f"      Aᵣ                      = {Ar:.3f} {AMPLITUDE_UNIT}")
print(f"  5)  Cycles (peak count)     = {cycles}")
print(f"  6)  Mean amplitude          = {mean_amp:.3f} {AMPLITUDE_UNIT}")
print(f"  7)  Std dev amplitude       = {std_amp:.3f} {AMPLITUDE_UNIT}")
print(f"  8)  Dominant FFT freq       = {fft_f:.3f} {FREQ_UNIT}")
print(f"  9)  Dominant FFT magnitude  = {fft_M:.3f}")
print(f" 10)  Distance to defect      = {dist_out:.3f} {DISTANCE_UNIT}")


In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks
from scipy.fft import fft, fftfreq

# ─── User settings ────────────────────────────────────────────────
FILENAME   = '/content/A5 S1C-09 Casing-Insulation Debond G-22-23 Rx.csv'
TIME_UNIT  = 'ms'     # your CSV's time unit: 'µs', 'ms', or 's'
AMPL_UNIT  = 'V'      # amplitude unit, e.g. 'V' or 'a.u.'
TIME_SCALE = 1e-3     # 1e-6 for µs, 1e-3 for ms, 1.0 for s

# ─── 1) Load data and convert ──────────────────────────────────────
df      = pd.read_csv(FILENAME)
t_raw   = df['time'].values
amp     = df['amplitude'].values
t       = t_raw * TIME_SCALE  # seconds

# ─── 2) Peak extraction helper ────────────────────────────────────
def find_peak_in_window(t, a, tmin, tmax):
    mask = (t>=tmin) & (t<=tmax)
    if not mask.any():
        return np.nan, np.nan
    subset = a[mask]
    idx    = np.argmax(subset)
    return float(t[mask][idx]), float(subset[idx])

# ─── 3) Compute the 12 features ───────────────────────────────────
T0 = 0.0

# windows defined in original units → convert to seconds
w1 = (140*TIME_SCALE, 260*TIME_SCALE)
w2 = (300*TIME_SCALE, 400*TIME_SCALE)
w3 = (400*TIME_SCALE, 600*TIME_SCALE)

T1, A1 = find_peak_in_window(t, amp, *w1)
Tp, Ap = find_peak_in_window(t, amp, *w2)
Tr, Ar = find_peak_in_window(t, amp, *w3)

# cycle count (peaks above mean)
mean_amp = amp.mean()
peaks, _ = find_peaks(amp, height=mean_amp)
cycles   = len(peaks)

std_amp  = amp.std()

# FFT for dominant frequency & magnitude
N   = len(amp)
dt  = t[1] - t[0]
Y   = fft(amp - mean_amp)
F   = fftfreq(N, dt)
pos = F>0
freqs_hz = F[pos]
mags     = np.abs(Y[pos])

dom = np.argmax(mags)
fft_freq_khz = freqs_hz[dom]/1e3
fft_mag      = float(mags[dom])

# ─── 4) Rule-based classification ─────────────────────────────────
is_good = True
# check T1, Tp, Tr in their good ranges (in original units)
if not (140 <= T1/TIME_SCALE <= 260): is_good = False
if not (300 <= Tp/TIME_SCALE <= 400): is_good = False
if not (400 <= Tr/TIME_SCALE <= 600): is_good = False
# check FFT magnitude
if fft_mag >= 100:                  is_good = False

label = "Good" if is_good else "Defect"

# ─── 5) Print results ──────────────────────────────────────────────
print("Extracted Features:")
print(f" T0       = {T0:.3f} {TIME_UNIT}")
print(f" T1       = {T1/TIME_SCALE:.3f} {TIME_UNIT},  A1 = {A1:.3f} {AMPL_UNIT}")
print(f" Tp       = {Tp/TIME_SCALE:.3f} {TIME_UNIT},  Ap = {Ap:.3f} {AMPL_UNIT}")
print(f" Tr       = {Tr/TIME_SCALE:.3f} {TIME_UNIT},  Ar = {Ar:.3f} {AMPL_UNIT}")
print(f" Cycles   = {cycles}")
print(f" Mean amp = {mean_amp:.3f} {AMPL_UNIT}")
print(f" Std amp  = {std_amp:.3f} {AMPL_UNIT}")
print(f" FFT freq = {fft_freq_khz:.3f} kHz")
print(f" FFT mag  = {fft_mag:.3f}")
print()
print(f"→ Classification: **{label}**")


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.signal import hilbert, find_peaks
from scipy.fft import rfft, rfftfreq
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from IPython.display import display, Markdown, Javascript
try:
    import ipywidgets as widgets
    have_widgets = True
except ImportError:
    have_widgets = False

# --- CSV Scanning and Interactive Picker ---
def setup_csv_picker():
    """Scan for CSVs and provide an interactive picker."""
    csv_folder = os.getcwd()  # Default to current working directory
    try:
        all_csvs = sorted(
            os.path.join(csv_folder, fn)
            for fn in os.listdir(csv_folder)
            if fn.lower().endswith(".csv")
        )
    except FileNotFoundError:
        print(f"Directory {csv_folder!r} not found.")
        all_csvs = []
    except PermissionError:
        print(f"Permission denied accessing {csv_folder!r}.")
        all_csvs = []

    # Display CSV paths
    if all_csvs:
        display(Markdown("### Available CSV files (copy-&-paste these paths into `input_path`):"))
        for path in all_csvs:
            display(Markdown(f"`{path}`"))
    else:
        display(Markdown("**No CSV files found.**"))

    # Interactive picker
    if have_widgets and all_csvs:
        display(Markdown(
            "Select a CSV from the dropdown. The path appears in the text box. "
            "Click 'Copy Path' to copy the path to your clipboard, or use `out.value` in your code."
        ))
        dd = widgets.Dropdown(
            options=all_csvs,
            description="Pick CSV:",
            layout=widgets.Layout(width="80%")
        )
        out = widgets.Text(
            value=all_csvs[0],
            description="input_path:",
            layout=widgets.Layout(width="80%")
        )
        copy_button = widgets.Button(
            description="Copy Path",
            button_style="info",
            tooltip="Copy the selected path to clipboard"
        )

        def on_change(change):
            out.value = change.new
        def on_copy_button_clicked(b):
            # JavaScript to copy text to clipboard
            js_code = f"""
            navigator.clipboard.writeText("{out.value}");
            """
            display(Javascript(js_code))
            print("Path copied to clipboard!")

        dd.observe(on_change, names='value')
        copy_button.on_click(on_copy_button_clicked)
        display(dd, out, copy_button)
    else:
        if not have_widgets:
            print("Install ipywidgets (`pip install ipywidgets`) for a dropdown picker.")
        elif not all_csvs:
            print(f"No CSVs found in {csv_folder!r}.")

# --- Ultrasonic Analysis Functions ---
def compute_fft_features(time, amplitude):
    """Compute dominant frequency (Hz) and magnitude from FFT of the signal."""
    if len(time) < 2 or len(amplitude) < 2:
        return None, None
    try:
        dt = (time[1] - time[0]) * 1e-6  # time step in seconds
        if dt <= 0:
            return None, None
        signal = amplitude - np.mean(amplitude)
        N = len(signal)
        Y = rfft(signal)
        freqs = rfftfreq(N, dt)
        mag = np.abs(Y)
        if len(mag) > 0:
            mag[0] = 0  # ignore DC
        idx = np.argmax(mag)
        dom_freq = freqs[idx] if idx < len(freqs) else None
        dom_mag = mag[idx] if idx < len(mag) else None
        return dom_freq, dom_mag
    except Exception:
        return None, None

def extract_ultrasonic_features(time, amplitude):
    """Extract ultrasonic features for a given time (µs) and amplitude array."""
    time = np.array(time, dtype=float)
    amplitude = np.array(amplitude, dtype=float)
    features = {}
    if time.size == 0 or amplitude.size == 0:
        return features

    features["T0"] = float(time[0])
    abs_amp = np.abs(amplitude)
    Tp_idx = int(np.argmax(abs_amp))
    features["Tp"] = float(time[Tp_idx])
    features["Ap"] = float(amplitude[Tp_idx])

    tail = amplitude[int(0.9*len(amplitude)):] if len(amplitude) > 10 else amplitude
    noise_std = float(np.std(tail)) if tail.size > 0 else 0.0
    threshold = max(5 * noise_std, 0.05 * np.max(abs_amp))

    above_idx = np.where(abs_amp > threshold)[0]
    if above_idx.size > 0 and above_idx[0] == 0:
        above_idx = above_idx[1:]
    if above_idx.size > 0:
        T1_idx = int(above_idx[0])
        features["T1"] = float(time[T1_idx])
        features["T1_amp"] = float(amplitude[T1_idx])
    else:
        features["T1"] = None
        features["T1_amp"] = None

    envelope = np.abs(hilbert(amplitude))
    peak_indices, _ = find_peaks(envelope, height=threshold)
    peak_indices = sorted(peak_indices, key=lambda i: time[i])
    features["echo_count"] = len(peak_indices)
    features["echo_spacing"] = None
    if features["echo_count"] > 1:
        spacings = np.diff([time[i] for i in peak_indices])
        if spacings.size > 0:
            features["echo_spacing"] = float(np.mean(spacings))

    Tr_idx = Tp_idx
    if peak_indices:
        if Tp_idx in peak_indices:
            Tp_pos = peak_indices.index(Tp_idx)
        else:
            Tp_pos = 0
            for j, idx in enumerate(peak_indices):
                if abs(idx - Tp_idx) < 3:
                    Tp_pos = j
                    break
        if Tp_pos < len(peak_indices) - 1:
            Tr_idx = peak_indices[Tp_pos + 1]
    features["Tr"] = float(time[Tr_idx]) if Tr_idx is not None else None
    features["Ar"] = float(amplitude[Tr_idx]) if Tr_idx is not None else None

    features["SNR"] = float(np.max(envelope) / noise_std) if noise_std > 0 else float('inf')
    dom_freq, dom_mag = compute_fft_features(time, amplitude)
    features["dom_freq_kHz"] = float(dom_freq/1000.0) if dom_freq is not None else None
    features["dom_mag"] = float(np.abs(dom_mag)) if dom_mag is not None else None
    return features

def classify_good_or_defect(feat):
    """Apply criteria to classify as Good or Defect."""
    if (feat.get("T1") is None or feat["T1"] < 140 or feat["T1"] > 260 or
        feat.get("Tp") is None or feat["Tp"] < 300 or feat["Tp"] > 400 or
        feat.get("Tr") is None or feat["Tr"] < 400 or feat["Tr"] > 600 or
        feat.get("dom_mag") is None or feat["dom_mag"] >= 100 or
        feat.get("SNR") is None or feat["SNR"] < 10):
        return "Defect"
    return "Good"

def compute_defect_percentage(feat):
    """Compute a normalized defect percentage based on deviation from normal ranges."""
    deviations = []
    T1 = feat.get("T1")
    if T1 is None:
        deviations.append(1.0)
    else:
        if T1 < 140:
            deviations.append((140 - T1) / 140.0)
        elif T1 > 260:
            deviations.append((T1 - 260) / 260.0)
        else:
            deviations.append(0.0)
    Tp = feat.get("Tp")
    if Tp is None:
        deviations.append(1.0)
    else:
        if Tp < 300:
            deviations.append((300 - Tp) / 300.0)
        elif Tp > 400:
            deviations.append((Tp - 400) / 400.0)
        else:
            deviations.append(0.0)
    Tr = feat.get("Tr")
    if Tr is None:
        deviations.append(1.0)
    else:
        if Tr < 400:
            deviations.append((400 - Tr) / 400.0)
        elif Tr > 600:
            deviations.append((Tr - 600) / 600.0)
        else:
            deviations.append(0.0)
    dom_mag = feat.get("dom_mag")
    if dom_mag is None:
        deviations.append(1.0)
    else:
        deviations.append((min(dom_mag, 200) - 100) / 100.0 if dom_mag >= 100 else 0.0)
    SNR = feat.get("SNR")
    if SNR is None or SNR < 10:
        deviations.append((10 - (SNR or 0)) / 10.0)
    else:
        deviations.append(0.0)
    return float(min(np.mean(np.clip(deviations, 0, 1)) * 100.0, 100.0)) if deviations else 0.0

def predict_defect_type(feat, model, label_enc):
    """Use trained model to predict defect type and confidence."""
    feature_vector = np.array([[
        feat.get("T1", 0), feat.get("Tp", 0), feat.get("Tr", 0),
        feat.get("Ap", 0), feat.get("Ar", 0) if feat.get("Ar") is not None else 0,
        feat.get("echo_count", 0), feat.get("SNR", 0),
        feat.get("dom_freq_kHz", 0), feat.get("dom_mag", 0)
    ]])
    pred_idx = model.predict(feature_vector)[0]
    defect_type = label_enc.inverse_transform([pred_idx])[0]
    try:
        prob = model.predict_proba(feature_vector)[0]
        confidence = float(np.max(prob) * 100.0)
    except AttributeError:
        confidence = None
    return defect_type, confidence

def analyze_ultrasonic_data(input_path):
    """Analyze all CSV files in the given path (file or directory)."""
    files = []
    if os.path.isdir(input_path):
        try:
            for fname in os.listdir(input_path):
                if fname.lower().endswith(".csv"):
                    files.append(os.path.join(input_path, fname))
        except (FileNotFoundError, PermissionError) as e:
            print(f"Error accessing directory {input_path!r}: {e}")
            return []
    else:
        if not os.path.exists(input_path):
            print(f"File {input_path!r} does not exist.")
            return []
        files.append(input_path)

    training_features = []
    training_labels = []
    for file in files:
        name = os.path.basename(file).lower()
        label = None
        if "crack" in name:
            label = "Crack"
        elif "debond" in name:
            label = "Debonding"
        elif "composite defect" in name or "inclusion" in name:
            label = "Inclusion"
        elif "porosity" in name or "bonded zone" in name:
            label = "Porosity"
        if label:
            try:
                df = pd.read_csv(file)
                if 'time' not in df.columns or 'amplitude' not in df.columns:
                    print(f"Skipping {file}: Missing 'time' or 'amplitude' columns.")
                    continue
                feat = extract_ultrasonic_features(df['time'], df['amplitude'])
                vec = [
                    feat.get("T1", 0), feat.get("Tp", 0), feat.get("Tr", 0),
                    feat.get("Ap", 0), feat.get("Ar", 0) if feat.get("Ar") is not None else 0,
                    feat.get("echo_count", 0), feat.get("SNR", 0),
                    feat.get("dom_freq_kHz", 0), feat.get("dom_mag", 0)
                ]
                training_features.append(vec)
                training_labels.append(label)
            except Exception as e:
                print(f"Error processing {file} for training: {e}")
                continue

    rf_model = None
    label_enc = None
    if training_features and training_labels:
        try:
            label_enc = LabelEncoder()
            y = label_enc.fit_transform(training_labels)
            rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
            rf_model.fit(np.array(training_features), y)
        except Exception as e:
            print(f"Error training model: {e}")

    results = []
    for file in tqdm(files, desc="Analyzing files"):
        try:
            df = pd.read_csv(file)
            if 'time' not in df.columns or 'amplitude' not in df.columns:
                print(f"Skipping {file}: Missing 'time' or 'amplitude' columns.")
                continue
            feat = extract_ultrasonic_features(df['time'], df['amplitude'])
            classification = classify_good_or_defect(feat)
            defect_pct = compute_defect_percentage(feat)
            defect_type = "None"
            confidence = None
            explanation = ""
            remaining_life = ""
            if classification == "Defect":
                if rf_model and label_enc:
                    defect_type, confidence = predict_defect_type(feat, rf_model, label_enc)
                else:
                    defect_type = "Unknown"
                    confidence = None
                reasons = []
                if feat.get("T1") is None or feat["T1"] < 140 or feat["T1"] > 260:
                    reasons.append(f"T1={feat.get('T1') or 'None':.1f}µs out of range")
                if feat.get("Tp") is None or feat["Tp"] < 300 or feat["Tp"] > 400:
                    reasons.append(f"Tp={feat.get('Tp') or 'None':.1f}µs out of range")
                if feat.get("Tr") is None or feat["Tr"] < 400 or feat["Tr"] > 600:
                    reasons.append(f"Tr={feat.get('Tr') or 'None':.1f}µs out of range")
                if feat.get("dom_mag") is not None and feat["dom_mag"] >= 100:
                    reasons.append(f"FFT mag={feat['dom_mag']:.1f} ≥ 100")
                if feat.get("SNR") is not None and feat["SNR"] < 10:
                    reasons.append(f"SNR={feat['SNR']:.1f} (low)")
                explanation = " & ".join(reasons) if reasons else "Signal anomalies detected"
                if defect_type in ["Crack", "Debonding", "Delamination"]:
                    remaining_life = "Low (defect may propagate quickly)"
                elif defect_type in ["Porosity", "Inclusion", "Environmental Degradation"]:
                    remaining_life = "Moderate (monitor regularly)"
                else:
                    remaining_life = "Unknown"
            else:
                defect_type = "None"
                confidence = 100.0
                explanation = "All signals within normal range."
                remaining_life = "High (no significant defect)"

            sample_name = os.path.basename(file)
            print(f"\nSample: {sample_name}")
            print(f"  Classification: {classification}")
            print(f"  Defect percentage: {defect_pct:.1f}%")
            print(f"  Confidence: {confidence:.1f}%" if confidence is not None else "  Confidence: N/A")
            print(f"  Likely defect type: {defect_type}")
            print(f"  Explanation: {explanation}")
            print(f"  Estimated remaining life: {remaining_life}")
            results.append((sample_name, classification, defect_pct, confidence, defect_type, explanation, remaining_life))
        except Exception as e:
            print(f"Error processing {file}: {e}")
            continue
    return results

# --- Run CSV Picker and Example Analysis ---
if __name__ == "__main__":
    setup_csv_picker()
    # Example usage (uncomment and set your path):
    input_path = "/content/Propellant Crack A2-17 S-II E-7-8 Rx.csv"
    results = analyze_ultrasonic_data(input_path)